In [ ]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.trading.requests import GetAssetsRequest
import time
import datetime
from google.colab import userdata
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from alpaca.data.enums import DataFeed

In [ ]:
# ----------------- CONFIG -----------------
API_KEY = userdata.get('API_KEY')
SECRET_KEY = userdata.get('SECRET_KEY')
SYMBOL = "SPY"
INTERVAL_SECONDS = 120
RUN_DURATION_MINUTES = 20
RUNS = (RUN_DURATION_MINUTES * 60) // INTERVAL_SECONDS
QTY = 1

In [ ]:
# ----------------- API Clients -----------------
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)
trading_client = TradingClient(API_KEY, SECRET_KEY, paper=True)

In [ ]:
trades = []

In [ ]:
# ---------- Functions ----------
def validate_market_data(df):
    if df.empty:
        raise ValueError("❌ Market data is empty.")
    if (df['close'] <= 0).any():
        raise ValueError("❌ Close prices contain non-positive values.")
    if not df.index.is_monotonic_increasing:
        raise ValueError("❌ Timestamps are not in increasing order.")
    return True

def fetch_bars(client, symbol, start, end, tf_override):
    request = StockBarsRequest(
        symbol_or_symbols=[symbol],
        timeframe=TimeFrame.Minute,
        start=start,
        end=end,
        timeframe_override=tf_override,
        feed=DataFeed.IEX
    )
    bars = client.get_stock_bars(request)[symbol]
    df = pd.DataFrame([{
        "timestamp": bar.timestamp,
        "close": bar.close
    } for bar in bars])
    df.set_index("timestamp", inplace=True)
    validate_market_data(df)
    return df

def compute_trend_ratio(short_df, long_df):
    short_avg = short_df['close'].resample('15min').mean()
    long_avg = long_df['close']
    combined = pd.DataFrame({'short_avg': short_avg, 'long_avg': long_avg}).dropna()
    combined['trend_ratio'] = combined['short_avg'] / combined['long_avg']
    return combined[['trend_ratio']]

def generate_signal(df, trend_ratio_series):
    ema20 = df['close'].ewm(span=100).mean()
    ema50 = df['close'].ewm(span=200).mean()
    latest_ratio = trend_ratio_series['trend_ratio'].iloc[-1]
    if ema20.iloc[-1] > ema50.iloc[-1] and latest_ratio > 1.01:
        return "BUY"
    elif ema20.iloc[-1] < ema50.iloc[-1] and latest_ratio < 0.99:
        return "SELL"
    else:
        return "HOLD"

In [ ]:
# ---------- Main Loop ----------
print("📈 Starting 5-minute live backtest...\n")
position = 0
cash = 100_000
shares = 0
equity_curve = []

for i in range(RUNS):
    try:
        now = datetime.datetime.utcnow()
        start = now - datetime.timedelta(days=3)
        end = now

        # Fetch data
        short_df = fetch_bars(data_client, SYMBOL, start, end, tf_override=1)
        long_df = fetch_bars(data_client, SYMBOL, start, end, tf_override=2)
        trend_ratio = compute_trend_ratio(short_df, long_df)

        # Signal
        signal_df = long_df.copy()
        signal = generate_signal(signal_df, trend_ratio)
        price = long_df['close'].iloc[-1]

        # Simulate trade
        if signal == "BUY" and position == 0:
            shares = QTY
            cash -= price * QTY
            position = 1
        elif signal == "SELL" and position == 1:
            cash += price * shares
            shares = 0
            position = 0

        equity = cash + shares * price
        equity_curve.append(equity)

        trades.append({
            "timestamp": now,
            "signal": signal,
            "price": price,
            "position": position,
            "equity": equity
        })

        print(f"[{now.strftime('%H:%M:%S')}] Signal: {signal} | Price: {price:.2f} | Equity: {equity:.2f}")

    except Exception as e:
        print(f"❌ Error at iteration {i}: {e}")

    time.sleep(INTERVAL_SECONDS)

📈 Starting 5-minute live backtest...

[12:14:19] Signal: HOLD | Price: 632.35 | Equity: 100000.00
[12:16:19] Signal: HOLD | Price: 632.35 | Equity: 100000.00
[12:18:19] Signal: HOLD | Price: 632.35 | Equity: 100000.00
[12:20:19] Signal: HOLD | Price: 632.35 | Equity: 100000.00
[12:22:19] Signal: HOLD | Price: 632.35 | Equity: 100000.00
[12:24:20] Signal: HOLD | Price: 632.35 | Equity: 100000.00


KeyboardInterrupt: 

In [ ]:
# ---------- KPIs and Visualization ----------
results_df = pd.DataFrame(trades).set_index("timestamp")

# Returns
results_df["returns"] = results_df["equity"].pct_change()
cumulative_return = results_df["equity"].iloc[-1] / results_df["equity"].iloc[0] - 1
annualized_return = results_df["returns"].mean() * 252 * (60 / INTERVAL_SECONDS)
annualized_volatility = results_df["returns"].std() * np.sqrt(252 * (60 / INTERVAL_SECONDS))
sharpe_ratio = annualized_return / annualized_volatility if annualized_volatility > 0 else np.nan
drawdown = results_df["equity"] - results_df["equity"].cummax()
max_drawdown = drawdown.min()

# Win rate
trade_changes = results_df["position"].diff()
wins = (results_df["returns"] > 0) & (trade_changes != 0)
win_rate = wins.sum() / max(wins.count(), 1)

# Print KPIs
print("\n📊 KPIs:")
print(f"Cumulative Return:     {cumulative_return:.4f}")
print(f"Annualized Return:     {annualized_return:.4f}")
print(f"Annualized Volatility: {annualized_volatility:.4f}")
print(f"Sharpe Ratio:          {sharpe_ratio:.2f}")
print(f"Max Drawdown:          {max_drawdown:.4f}")
print(f"Win Rate:              {win_rate:.2%}")

# Plot
results_df["equity"].plot(figsize=(12, 5), title="📈 Equity Curve (5-min Live Backtest)")
plt.xlabel("Time")
plt.ylabel("Equity ($)")
plt.grid(True)
plt.tight_layout()
plt.show()